In [ ]:
import pandas as pd
import os
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
feat = pd.read_csv("/content/drive/MyDrive/BTT_Amazon1A_Studio/feat.csv")

In [ ]:
feat.head()

,Customer ID,SnapshotMonth,month_orders,month_revenue,month_quantity,month_diff_items,month_last_order,month_cancellations,cancellation_rate,churn,snapshot_end,last_order,first_order,total_orders,tenure_days,recency_days,frequency,orders_30d,orders_60d,orders_90d
0,12346,2009-12,2.0,113.50,26.0,2.0,2009-12-18 10:55:00,0.0,0.000000,0,2009-12-31,2009-12-18 10:55:00,2009-12-14 08:34:00,5.0,16.0,12.0,5.0,5.0,5.0,5.0
1,14606,2009-12,11.0,1401.59,527.0,210.0,2009-12-21 14:33:00,12.0,1.090909,0,2009-12-31,2009-12-21 14:33:00,2009-12-03 12:40:00,8.0,27.0,9.0,8.0,8.0,8.0,8.0
2,14605,2009-12,1.0,208.60,133.0,31.0,2009-12-08 13:59:00,0.0,0.000000,1,2009-12-31,2009-12-08 14:01:00,2009-12-08 13:59:00,2.0,22.0,22.0,2.0,2.0,2.0,2.0
3,14593,2009-12,1.0,391.68,157.0,65.0,2009-12-09 12:57:00,2.0,2.000000,0,2009-12-31,2009-12-09 12:57:00,2009-12-09 12:57:00,1.0,21.0,21.0,1.0,1.0,1.0,1.0
4,14590,2009-12,3.0,349.66,108.0,38.0,2009-12-16 11:26:00,0.0,0.000000,0,2009-12-31,2009-12-16 11:26:00,2009-12-07 14:10:00,3.0,23.0,14.0,3.0,3.0,3.0,3.0


In [ ]:
feat.columns

Index(['Customer ID', 'SnapshotMonth', 'month_orders', 'month_revenue',
       'month_quantity', 'month_diff_items', 'month_last_order',
       'month_cancellations', 'cancellation_rate', 'churn', 'snapshot_end',
       'last_order', 'first_order', 'total_orders', 'tenure_days',
       'recency_days', 'frequency', 'orders_30d', 'orders_60d', 'orders_90d'],
      dtype='object')

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve, auc

FEATURES = ["cancellation_rate","total_orders", "tenure_days",
            "recency_days", "frequency", "orders_30d", "orders_60d",
            "orders_90d", "month_orders", "month_revenue", "month_quantity",
            "month_diff_items", "month_cancellations"]

X = feat[FEATURES]
y = feat['churn']

#train and test splits
X_train,X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

Random forest implementation

In [ ]:
#20 decision trees
#create model object and fit to training data
rf_20_model = RandomForestClassifier(criterion='entropy', n_estimators=20, random_state=42)
rf_20_model.fit(X_train, y_train)

#gets the predicted probability that each customer will churn
rf_20_preds = rf_20_model.predict_proba(X_test)
rf_20_predictions = list(rf_20_preds[:, -1])

#80 decision trees
#create model object and fit to training data
rf_80_model = RandomForestClassifier(criterion='entropy', n_estimators=80, random_state=42)
rf_80_model.fit(X_train, y_train)

#gets the predicted probability that each cutomer will churn
rf_80_preds = rf_80_model.predict_proba(X_test)
rf_80_predictions = list(rf_80_preds[:, -1])


In [ ]:
from sklearn.metrics import accuracy_score
#predict class labels for test data
rf_20_pred = rf_20_model.predict(X_test)
rf_80_pred = rf_80_model.predict(X_test)

#calculate accuracy scores
rf_20_acc = accuracy_score(y_test, rf_20_pred)
rf_80_acc = accuracy_score(y_test, rf_80_pred)

#print results
print(f"20 trees accuracy score: {rf_20_acc}")
print(f"80 trees accuracy score: {rf_80_acc}")

20 trees accuracy score: 0.7997159090909091
80 trees accuracy score: 0.8006628787878788


In [ ]:
from sklearn.model_selection import GridSearchCV

#hyperparameter tuning
param_grid = {
    "min_samples_leaf": [1, 5, 10, 50, 100],
    "max_depth": [4, 8, 16, 20, 24],
    "min_samples_split": [2, 5, 10, 20, 50]
}

grid_search = GridSearchCV(
    estimator = RandomForestClassifier(random_state=42, criterion='entropy'),
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print(f"hyperparameters: {grid_search.best_params_}")
print(f"best validation f1 score: {grid_search.best_score_:.4f}")

hyperparameters: {'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 5}
best validation f1 score: 0.4949


In [ ]:
#train model with best hyperparameters
rf_tuned_model = RandomForestClassifier(criterion='entropy', n_estimators=80, random_state=42,
                                     max_depth=20, min_samples_leaf=1, min_samples_split=5)
rf_tuned_model.fit(X_train, y_train)

#gets the predicted probability that each cutomer will churn
rf_tuned_preds = rf_tuned_model.predict_proba(X_test)
rf_tuned_predictions = list(rf_tuned_preds[:, -1])

#predict class labels for test data
rf_tuned_pred = rf_tuned_model.predict(X_test)

#calculate accuracy scores
rf_tuned_acc = accuracy_score(y_test, rf_tuned_pred)


In [ ]:
#print results
print(f"tuned hyperparameter accuracy score: {rf_tuned_acc}")

tuned hyperparameter accuracy score: 0.8011363636363636
